# BC–WA PM2.5 benchmark on Kaggle

**Settings (right panel):** Accelerator **GPU T4 x2** · Internet **On** (phone-verified account).
Then **Save Version → Save & Run All** - it runs in the background (browser can be closed) and
everything in `/kaggle/working/metrics` is saved as this version's output.

One version = one model group. Both T4s are used: the group's models (or, for a one-model group,
its 5 runs) are split over the two GPUs, which roughly halves the wall time for the same GPU quota.

| Group | Models |
|---|---|
| 1 | MLP, GRU, LSTM, PM25_GNN, AirLapseV2, Crossformer, Transformer, AirDualODE |
| 2 | STMamba, Informer, AirFormer |
| 3 | TCN_DIR, Autoformer |
| 4 | AirDDE (slow ODE model) |
| 5 | AirPhyNet (slow ODE model, ~3.3 h per run on a T4) |

**Staying inside your GPU quota:** set `GPU_HOURS_LEFT` to what Kaggle shows as remaining. Training
gets a hard deadline 20 min before that (and before Kaggle's 12 h limit): a run still going at the
deadline is stopped and redone next time, every finished run is kept, and the version ends cleanly.

To continue a group or combine groups, add earlier versions' output as input
(**Add Input → Notebooks → this notebook → that version**); their score files are picked up automatically.

In [ ]:
GROUP = 3              # 1-5, see the table above
REPEATS = 5
GPU_HOURS_LEFT = 6.0   # remaining Kaggle GPU quota this week (Settings -> your quota)

In [ ]:
import glob, os, shutil, subprocess, sys, time, torch
T_START = time.time()
N_GPU = torch.cuda.device_count()
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(N_GPU)] or 'NONE - set Accelerator to GPU T4 x2')

!git clone -q https://github.com/Grace-VN/BC-WA-PM2.5-Forecast.git /tmp/repo
%cd /tmp/repo
!pip install -q -r requirements.txt

METRICS = '/kaggle/working/metrics'          # saved as notebook output
os.makedirs(f'{METRICS}/scores', exist_ok=True)
os.environ['METRICS_DIR'] = METRICS
os.environ['RESULTS_DIR'] = '/tmp/results'   # large raw arrays: not saved

# resume / combine: copy score files from any attached earlier outputs
found = glob.glob('/kaggle/input/**/scores/*.csv', recursive=True)
for f in found:
    shutil.copy(f, f'{METRICS}/scores/')
print(f'{len(found)} earlier run results picked up')

In [ ]:
sys.path.insert(0, '/tmp/repo')
from benchmark import GROUPS
models = GROUPS[GROUP]
n = max(N_GPU, 1)
if len(models) >= n:                     # several models: spread the models over the GPUs
    streams = [dict(models=models[i::n]) for i in range(n)]
else:                                    # one model: spread its runs over the GPUs
    streams = [dict(models=models, repeat_ids=list(range(REPEATS))[i::n]) for i in range(n)]

# hard deadline for training, counted from now: 20 min before the quota runs out
# (or before Kaggle's 12 h session limit), minus the time setup already used
used_h = (time.time() - T_START) / 3600
deadline = min(GPU_HOURS_LEFT, 12.0) - 20 / 60 - used_h
print(f'training deadline: {deadline:.2f} h from now')

procs = []
for i, st in enumerate(streams):
    cmd = [sys.executable, 'benchmark.py', '--models', *st['models'], '--repeats', str(REPEATS),
           '--epochs', '50', '--early_stop', '10', '--deadline_hours', f'{deadline:.3f}']
    if N_GPU:
        cmd += ['--gpu', str(i)]
    if 'repeat_ids' in st:
        cmd += ['--repeat_ids', *map(str, st['repeat_ids'])]
    print(f'stream {i}:', ' '.join(cmd[2:]))
    procs.append(subprocess.Popen(cmd, stdout=open(f'{METRICS}/stream{i}.log', 'w'),
                                  stderr=subprocess.STDOUT))

def progress():
    for i in range(len(procs)):
        lines = [l.rstrip() for l in open(f'{METRICS}/stream{i}.log') if l.startswith(('==', '   '))]
        print(f'[{(time.time() - T_START) / 3600:.1f} h] stream {i}:', lines[-1] if lines else 'starting')

while any(p.poll() is None for p in procs):
    time.sleep(600)
    progress()
progress()

In [ ]:
# everything available in this version (this group + attached outputs): mean ± std
!python benchmark.py --summary
import pandas as pd
s = pd.read_csv(f'{METRICS}/benchmark_summary.csv', index_col=0)
cols = ['RMSE', 'MAE', 'MAPE', 'RMSE_gt35.5', 'CSI@35.5', 'POD@35.5', 'FAR@35.5', 'CSI@55.5']
table = pd.DataFrame({'runs': s.n_repeats})
for c in cols:
    d = 1 if c == 'MAPE' else 3
    table[c] = [f'{m:.{d}f} ± {sd:.{d}f}' if pd.notna(sd) else f'{m:.{d}f}'
                for m, sd in zip(s[c + '_mean'], s[c + '_std'])]
table